# Ultimate Guide to Web Scraping with Python Part 1: Requests and BeautifulSoup

In this tutorial, I will learn how to
- Request web pages
- Parse HTML
- Save and load scraped data
- Scrape multiple pages in a row.

## 1. Limit your impact when scraping
Every time you load a web page, you're making a request to a server.
With a Python script that can execute thousands of requests a second, 
if coded incorrectly, you could end up costing the website owner money and 
possibly bring down their site (DDoS attack).

**Every time we scrape, we should make only one request per page.**

_Hence why I shifted to Jupyter Notebook right here._

### Save HTML Locally

In [54]:
# Function to save HTML web content locally
def save_html(html, path):
    with open(path, 'wb') as f: 
        # 'wb' means "write bytes" - avoids encoding issues
        f.write(html)

# save_html(r.content, 'google_com')
# Assuming r.content is HTML from google.com 
# We now have a "google_com" file that contains the HTML from google.com

In [55]:
# Function to open/read HTML from local file
def open_html(path):
    with open(path, 'rb') as f:
        # 'rb' means "read bytes"
        return f.read()
    
# html = open_html('google_com')
# Reads HTML from file "google_com"

# If our script fails / computer shuts down - we no longer
# need to request Google again, lessening the impact on their servers

**Advice**

Save every page you need and parse later when web scraping as a safety precaution.

### Scrapers and Bots

Each site usually has a robots.txt on the root of their domain, which explicitly states what bots are allowed to do on their site, for example:

**Note** : `robots.txt` works by exclusion. Anything **not explicitly disallowed is implicitly allowed**.

## 2. Scraping Project: Getting Media Bias Data

`save_html(r.content, 'google_com')`

With Python's `requests`library, we get a web page by using `get()` on the URL. The response is contained in `r`, which contains many things, but `r.content` gives us the HTML.

Once we have the HTML, we can save it to a file and parse it for the data we're interested in. 

In this project, we are scraping the _AllSides_ website, which has a media bias rating table.

In [ ]:
import requests # the classic way
import pprint

url = 'https://www.allsides.com/media-bias/media-bias-ratings'

r = requests.get(url)

# Print the first 100 chars to confirm we have the source of the page
pprint.pprint(r.content[:100])

403
<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scale=1"><style>*{box-sizing:border-box;margin:0;padding:0}html{line-height:1.15;-webkit-text-size-adjust:100%;color:#313131;font-family:system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,"Helvetica Neue",Arial,"Noto Sans",sans-serif,"Apple Color Emoji","Segoe UI Emoji","Segoe UI Symbol","Noto Color Emoji"}body{display:flex;flex-direction:column;height:100vh;min-height:100vh}.main-content{margin:8rem auto;padding-left:1.5rem;max-width:60rem}@media (width <= 720px){.main-content{margin-top:4rem}}.h2{line-height:2.25rem;font-size:1.5rem;font-weight:500}@media (width <= 720px){.h2{line-height:1.5rem;font-size:1.25rem}}#challenge-error-text{background-image:url("data:image/svg+xml;base

Because we're being blocked by CloudFlare, we can download a scraping-friendly library that can handle CloudFlare, e.g. `cloudscraper`.

In [89]:
import cloudscraper

scraper = cloudscraper.create_scraper()

url = 'https://www.allsides.com/media-bias/media-bias-ratings'

r = scraper.get(url)

# Print the first 100 chars to confirm we have the source of the page
# pprint.pprint(r.content[:1000])

In [92]:
# Save the HTML to a file so you don't need to keep get()-ing the server. 
# Which is probably why they put up the CloudFlare...
save_html(r.content, 'AllSides_html')
html = open_html('AllSides_html')

### Parsing HTML with BeautifulSoup
_My soup is beautiful._

We have obtained the HTML from the _AllSides_ server, but now we need to parse the HTML with BeautifulSoup.

When we pass HTML to the BeautifulSoup constructor, this returns an object that we can then navigate like the original tree structure of the Document Object Model (DOM). _Side note: the DOM represents a web page as a tree-like structure of elements._

**insert image of a DOM here**

This way we can find elements using names of tags, classes, IDs, and through relationships to other elements, like getting the children and siblings of elements.

In [94]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, 'html.parser')

print(soup)

<!DOCTYPE html>
<html data-astro-cid-sckkx6r4="" lang="en"> <head><script>(function(w,i,g){w[g]=w[g]||[];if(typeof w[g].push=='function')w[g].push(i)})
(window,'GTM-KLVCVB2','google_tags_first_party');</script><script>(function(w,d,s,l){w[l]=w[l]||[];(function(){w[l].push(arguments);})('set', 'developer_id.dY2E1Nz', true);
		var f=d.getElementsByTagName(s)[0],
		j=d.createElement(s);j.async=true;j.src='/rp9j/';
		f.parentNode.insertBefore(j,f);
		})(window,document,'script','dataLayer');</script><!-- Google Tag Manager --><script type="086d451aaf5516ceb5bb3f03-module">function l(){const e=(...a)=>{};(()=>{try{const a=window.self!==window.top;return e("[IframeNav] In iframe check:",a),a}catch{return!0}})()&&document.addEventListener("click",a=>{const o=a.target?.closest("a");if(o){const n=o.getAttribute("href")||"";let r=n;try{const t=new URL(n,window.location.origin);r=`${t.pathname}${t.search}${t.hash}`,r.endsWith("/index.html")&&(r=r.replace("/index.html","")),e("[IframeNav] Normaliz

### Making sense of the HTML - finding elements and data
To find the elements and data inside our HTML, we will use:
- `select_one` - this returns a single element.
- `select` - this returns a list of elements, even if only one item exists. 

Both methods use CSS selectors to find elements. 

### Ok, what the hell is a CSS selector?
_My thanks to GPT for helping explain this._

CCS stands for "Cascading Style Sheet". Cascading Style Sheets is what makes a website _look nice_! It controls aspects like:
- Font size
- Layout and spacing
- Backgrounds
- Positions of elements/pictures.

HTML builds the **structure**, CSS adds the **style**.

#### Nice. So what are tags? And why do I care?
Tags in CSS are usually **HTML tags**, also called **elements** - what you want to add style to. 

**NOTE**: When you change the cell language to HTML in Jupyter Notebook, **it only changes syntax highlighting, not execution**.

Jupyter itself still tries to execute the contents of the cell as Python, unless you explicitly tell it otherwise.

So even if the dropdown says “HTML”, Jupyter doesn’t know to render HTML — it still tries to run it as code and fails. 
To get around it, put `%%html` before the HTML code, to use the "magic command".

For example, in HTML, you might have:

In [58]:
%%html
<p>This is a paragraph.</p>
<h1>This is a heading.</h1>
<button>Click me!</button>


`<p>` is a tag/element for a paragraph.

In CSS, you "target" these tags to style them.

#### How CSS uses tags
Below, in CSS:

This means:
- All `<p>` elements - all paragraphs are now in blue
- All `<h1>` elements - all headings are 32 px big
- All `<button>` elements have a light grey background.

#### Basic Guide to CSS tags / elements and syntax

- `<a>` indicates the start of the element.
- `</a>` closes the element.

Below is copied shamelessly from learndatasci.com.

1. To get a tag/element, like `<a></a>` - use its naked name e.g. `select_one('a')` or `select_one('body')`.
2. `.temp` gets an element with a class of **temp**, e.g. to get `<a class="temp"></a>`, use `select_one('.temp')`.
3. `#temp` gets an element with an ID of **temp**, e.g. to get a `<a id="temp"></a>`, use `select_one('#temp')`.
4. `.temp.example` gets an element with both classes **temp** and **example** e.g. to get `<a class="temp example"></a>` use `select_one('.temp.example')`
5. `.temp a` gets an anchor element nested inside a parent element with class **temp** e.g. to get `<div class="temp"><a></a></div>` use `select_one('.temp a')`. Note the space between `.temp` and `a`. 
6. `.temp .example` gets an element with class example nested inside of a parent element with class temp, E.g. to get `<div class="temp"><a class="example"></a></div>` use `select_one('.temp .example')`. Again, note the space between .temp and .example. The space tells the selector that the class after the space is a child of the class before the space.
7. ids, such as `<a id=one></a>`, are unique so you can usually use the id selector by itself to get the right element. No need to do nested selectors when using ids.

_Other selectors are available_ (on Google).


#### Tips on figuring out how to select certain elements

You can find the selector for an element in a browser's developer tools: 
1. In Chrome, this is right-click and "Inspect". This highlights the element you right-clicked.
2. Right-click the code element in dev. tools, hover over "Copy" which takes you to a dropdown, then "Copy element".


### Let's Start

Our data is housed in a table on AllSides. By inspecting the header element of the table, we can find the code that renders the table and the rows.

_I want to say it's `<thead>`, which leads on to `<table class="w-full">`_.

Copying the element `<table class="w-full">` gives (massively truncated):

In [96]:
%%html
<table class="w-full">
    <thead>
        <tr class="border-b-[3px] border-color-lightgrey">
            <th class="text-left py-2">News Source</th>
            <th class="text-left py-2 hidden md:table-cell">Type</th>
            <th class="text-left py-2 md:px-4 px-2 w-[120px] md:w-[180px]">AllSides Bias Rating</th>
            <th class="text-left py-2 hidden md:table-cell md:w-[200px]">What do you think?</th>
        </tr>
    </thead>
    <tbody> 
</table>

News Source,Type,AllSides Bias Rating,What do you think?


Simplifying the HTML structure:

In [97]:
%%html
<table>
    <thead> 
        <!-- header information --> 
    </thead>
    <tbody>
        <tr class="odd views-row-first">                                       <!-- begin table row -->
            <td class="views-field views-field-title source-title">            <!-- table cell -->
                <!-- outlet name -->
            </td> 
            <td class="views-field views-field-field-bias-image">              <!-- table cell -->
                <!-- bias data -->
            </td> 
            <td class="views-field views-field-nothing-1 what-do-you-think">   <!-- table cell -->
                <!-- agree / disagree buttons -->
            </td> 
            <td class="views-field views-field-nothing community-feedback">    <!-- table cell -->
                <!-- agree / disagree data -->
            </td> 
        </tr>                                                                  <!-- end table row -->
        <!-- more rows -->
    </tbody>
</table>

<!-- table cell --> <!-- outlet name -->,<!-- table cell --> <!-- bias data -->,<!-- table cell --> <!-- agree / disagree buttons -->,<!-- table cell --> <!-- agree / disagree data -->


Therefore, to get each row, we just need to select all the `<tr>`(**t**able **r**ows) inside `<tbody>`.

In [98]:
rows = soup.select('table tbody tr')

# tbody tr tells selector to extract all the <tr> (table row) tags that are 
# children of the <tbody> tag.

# pprint.pprint(rows)

**NOTE**: If there was more than one table on this page, we would have to make a more specific selector. But as this is the only table, it's fine.

Now, we have a list of HTML table rows in `rows` that each contain four cells, that correspond to the column names:
- News source name and link
- Bias data
- Agreement buttons
- Community feedback data

In [99]:
row = rows[0]
pprint.pprint(rows)

[<tr class="border-b border-color-lightgrey hover:bg-gray-50"><td class="py-3"><a class="text-color-link hover:underline" href="/news-source/abc-news-media-bias">ABC News (Online)</a></td><td class="py-3 md:px-4 px-2 text-[#0091FF] w-[120px] md:w-[180px]"><a href="/media-bias/left-center"><img alt="Lean Left" height="15" src="https://img.allsides.com/sites/default/files/bias-leaning-left.png" width="90"/></a></td><td class="py-3 md:whitespace-nowrap md:w-[200px]"><div class="flex gap-2 relative"><button class="md:px-4 md:py-1 bg-[#D0F1DF] text-[#007A25] rounded-full flex items-center justify-center w-7 md:w-24 hover:opacity-70 h-7"><i class="md:mr-1 fa-solid fa-check"></i><span class="hidden md:inline">agree</span></button><button class="md:px-4 md:py-1 bg-[#FBDADA] text-[#C21E1E] rounded-full flex items-center justify-center w-7 md:w-24 hover:opacity-70 h-7"><i class="md:mr-1 fa-solid fa-xmark"></i><span class="hidden md:inline">disagree</span></button></div></td></tr>,
 <tr class="bo